In [9]:
from pathlib import Path
import subprocess
import py3Dmol
from ipywidgets import interact, IntSlider, Dropdown
from molgeom import Molecule


def load_xyz_files(directory_path):
    """
    Preload all .xyz files in the directory and store their contents in a list.
    """
    directory_path = Path(directory_path)
    files = sorted(directory_path.glob("*.xyz"))
    xyz_contents = [file.read_text() for file in files]
    return xyz_contents, files


def update_structure(view, xyz_content, style, camera_state=None):
    """
    Update the molecular structure and style in an existing viewer.
    """
    view.removeAllModels()  # Clear the existing model
    view.addModel(xyz_content, "xyz")
    if style == "ball and stick":
        view.setStyle({"stick": {"radius": 0.1}, "sphere": {"radius": 0.3}})
    elif style == "stick":
        view.setStyle({"stick": {}})
    elif style == "sphere":
        view.setStyle({"sphere": {}})
    if camera_state:
        # Apply saved camera state
        view.setView(camera_state)
    else:
        # Default zoom if no camera state is provided
        view.zoomTo()
    view.update()  # Efficiently update the viewer without full re-rendering


def save_html(view, filename="viewer_output1.html"):
    """
    Save the generated HTML and JavaScript for the viewer to a file.
    """
    html_content = view._make_html()  # Generate the HTML content
    output_path = Path(filename)
    output_path.write_text(html_content, encoding="utf-8")
    print(f"HTML and JavaScript saved to {output_path}")


def create_interactive_visualizer(mols: list[Molecule], save_to_file: bool = False):
    """
    Create an interactive visualization for a directory of .xyz files.
    """
    # xyz_contents, files = load_xyz_files(directory_path)
    xyz_contents = [f"{len(mol)}\ncomment line\n{mol.to_xyz()}" for mol in mols]
    if not xyz_contents:
        raise FileNotFoundError("No .xyz files found in directory!")

    # Initialize viewer (only once)
    view = py3Dmol.view(width=800, height=400)
    view.show()  # Show the viewer only once at the beginning
    camera_state = None

    def update_view(idx, style):
        nonlocal camera_state
        xyz_content = xyz_contents[idx - 1]
        update_structure(view, xyz_content, style, camera_state)

    max_steps = len(xyz_contents)
    idx_slider = IntSlider(min=1, max=max_steps, step=1, value=1, description="Step")
    style_dropdown = Dropdown(
        options=["ball and stick", "stick", "sphere"], value="ball and stick", description="Style"
    )
    interact(update_view, idx=idx_slider, style=style_dropdown)

    if save_to_file:
        save_html(view)  # Save the HTML and JavaScript to a file

In [10]:

git_root = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
    ).stdout.strip()
)
dev_dir = git_root / "dev_dir"
files_path = git_root / "tests" / "files"
mols = []
mols.append(Molecule.from_file(files_path / "H2O.xyz"))
for i in range(1, 10):
    new_mol = mols[-1].copy()
    new_mol[0].translate([0.2, 0, 0])
    mols.append(new_mol)

# Create the interactive visualizer and save the HTML output
create_interactive_visualizer(mols, save_to_file=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

interactive(children=(IntSlider(value=1, description='Step', max=10, min=1), Dropdown(description='Style', opt…

HTML and JavaScript saved to viewer_output1.html


In [11]:
from pathlib import Path


def generate_html_with_widgets(mols, output_file="interactive_viewer_fixed.html"):
    """
    Generate an HTML file with 3Dmol.js and interactive widgets (slider and dropdown).
    """
    # Prepare XYZ contents
    xyz_contents = [f"{len(mol)}\ncomment line\n{mol.to_xyz()}" for mol in mols]

    # Build interactive HTML with widgets
    html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <title>3Dmol.js Interactive Viewer</title>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/3Dmol/2.4.2/3Dmol-min.js"></script>
</head>
<body>
    <div>
        <label for="step-slider">Step:</label>
        <input id="step-slider" type="range" min="1" max="{len(xyz_contents)}" value="1" step="1" 
               oninput="updateViewer()" style="width: 300px;">
        <label id="step-label">1</label>
    </div>
    <div>
        <label for="style-dropdown">Style:</label>
        <select id="style-dropdown" onchange="updateViewer()">
            <option value="ball and stick" selected>Ball and Stick</option>
            <option value="stick">Stick</option>
            <option value="sphere">Sphere</option>
        </select>
    </div>
    <div id="3dmolviewer"  style="position: relative; width: 800px; height: 400px;">
    </div>
    <script>
        var xyzData = {xyz_contents};
        var viewer = $3Dmol.createViewer(document.getElementById("3dmolviewer"), {{ backgroundColor: "white" }});

        function updateViewer() {{
            var step = document.getElementById("step-slider").value;
            var style = document.getElementById("style-dropdown").value;
            document.getElementById("step-label").innerText = step;

            viewer.removeAllModels();
            viewer.addModel(xyzData[step - 1], "xyz");
            if (style === "ball and stick") {{
                viewer.setStyle({{"stick": {{"radius": 0.1}}, "sphere": {{"radius": 0.3}}}});
            }} else if (style === "stick") {{
                viewer.setStyle({{"stick": {{}}}});
            }} else if (style === "sphere") {{
                viewer.setStyle({{"sphere": {{}}}});
            }}
            viewer.zoomTo();
            viewer.render();
        }}

        // Initialize the viewer with the first structure
        updateViewer();
    </script>
</body>
</html>
    """

    # Save the HTML file
    output_path = Path(output_file)
    output_path.write_text(html_template, encoding="utf-8")
    print(f"Interactive HTML saved to: {output_path}")



In [4]:
!pip install IPython

In [ ]:
# import webbrowser
from Ipython.display import HTML

git_root = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
    ).stdout.strip()
)
dev_dir = git_root / "dev_dir"
files_path = git_root / "tests" / "files"
mols = []
mols.append(Molecule.from_file(files_path / "H2O.xyz"))
for i in range(1, 10):
    new_mol = mols[-1].copy()
    new_mol[0].translate([0.2, 0, 0])
    mols.append(new_mol)

# Generate interactive HTML
generate_html_with_widgets(mols, output_file="interactive_viewer2.html")
display(HTML("interactive_viewer2.html"))
# Open the HTML file in the default web browser
# webbrowser.open("interactive_viewer2.html")

ModuleNotFoundError: No module named 'Ipython'